# Variance vs Base Rate

本笔记探索：基础爆率变化对出金抽数的均值/标准差/方差的影响（可设置保底）。
下方分为函数单元与参数运行单元，按需修改 `rates` / `runs` / `pity` / `seed` / `outdir`。


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


def simulate_one_cycle(base_rate: float, pity: int, rng: np.random.Generator) -> int:
    count = 0
    while True:
        count += 1
        if count >= pity:
            return count
        if rng.random() < base_rate:
            return count


def simulate_many(base_rate: float, pity: int, n_runs: int, seed: int):
    rng = np.random.default_rng(seed)
    data = np.empty(n_runs, dtype=np.int32)
    for i in range(n_runs):
        data[i] = simulate_one_cycle(base_rate, pity, rng)
    mean_v = float(np.mean(data))
    std_v = float(np.std(data, ddof=1))
    var_v = float(np.var(data, ddof=1))
    return mean_v, std_v, var_v


In [ ]:
# 参数：可调爆率列表、仿真次数、保底、种子、输出目录
rates = [0.006, 0.008, 0.010, 0.012, 0.016, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080]
runs = 30000
pity = 90
seed = 2026
outdir = "figures_notebook"

os.makedirs(outdir, exist_ok=True)
sns.set(style='whitegrid')

means, stds, vars_ = [], [], []
for idx, rate in enumerate(rates):
    seed_i = seed + idx * 17
    m, s, v = simulate_many(rate, pity, runs, seed_i)
    means.append(m)
    stds.append(s)
    vars_.append(v)
    print(f"rate={rate:.3%} mean={m:.2f} std={s:.2f} var={v:.2f}")

rates_pct = [r * 100 for r in rates]

plt.figure(figsize=(9, 5.5))
plt.plot(rates_pct, stds, marker='o', color='#1d91c0', label='Std of pulls')
plt.plot(rates_pct, np.sqrt(vars_), marker='x', linestyle='--', color='#fb6a4a', alpha=0.6, label='Std (from variance)')
plt.xlabel('Base 5-star rate (%)')
plt.ylabel('Std of pulls to 5-star')
plt.title(f'Standard deviation vs base rate (pity={pity}, runs={runs})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'variance_vs_rate_std.png'), dpi=220)
plt.close()

plt.figure(figsize=(9, 5.5))
plt.plot(rates_pct, vars_, marker='s', color='#225ea8', label='Variance of pulls')
plt.xlabel('Base 5-star rate (%)')
plt.ylabel('Variance of pulls to 5-star')
plt.title(f'Variance vs base rate (pity={pity}, runs={runs})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(outdir, 'variance_vs_rate_var.png'), dpi=220)
plt.close()

print(f"Saved plots to: {outdir}")


In [ ]:
# [Configuration] Simulation parameters
rates = [0.006, 0.008, 0.010, 0.012, 0.016, 0.020, 0.030, 0.040, 0.050, 0.060, 0.080]
runs = 30000
pity = 90
seed = 2026

sns.set(style='whitegrid')

# [Simulation] Iterate through different base rates
means, stds, vars_ = [], [], []
for idx, rate in enumerate(rates):
    seed_i = seed + idx * 17
    m, s, v = simulate_many(rate, pity, runs, seed_i)
    means.append(m)
    stds.append(s)
    vars_.append(v)
    print(f"rate={rate:.3%} mean={m:.2f} std={s:.2f} var={v:.2f}")

# [Conversion] Convert rates to percentage for display
rates_pct = [r * 100 for r in rates]

# [Visualization] Standard deviation vs base rate
plt.figure(figsize=(9, 5.5))
# [Comparison] Plot both direct std calculation and sqrt of variance for verification
plt.plot(rates_pct, stds, marker='o', color='#1d91c0', label='Std of pulls')
plt.plot(rates_pct, np.sqrt(vars_), marker='x', linestyle='--', color='#fb6a4a', alpha=0.6, label='Std (from variance)')
plt.xlabel('Base 5-star rate (%)')
plt.ylabel('Std of pulls to 5-star')
plt.title(f'Standard Deviation vs Base Rate (pity={pity}, runs={runs})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# [Visualization] Variance vs base rate
plt.figure(figsize=(9, 5.5))
# [Analysis] Display variance trend across different acquisition probabilities
plt.plot(rates_pct, vars_, marker='s', color='#225ea8', label='Variance of pulls')
plt.xlabel('Base 5-star rate (%)')
plt.ylabel('Variance of pulls to 5-star')
plt.title(f'Variance vs Base Rate (pity={pity}, runs={runs})')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("Variance analysis visualizations displayed successfully")